# 💻 Hands-On Lab Part 2 — Spectral Indices (45 Minutes)

Now that we can programmatically stream and isolate custom spectral bands precisely over our Area of Interest (AOI), we will move from qualitative viewing to quantitative analysis. 

In this lab, we will write custom matrix operations to calculate the **Normalized Difference Vegetation Index (NDVI)** and the **Normalized Difference Red Edge index (NDRE)**. These metrics allow digital agronomists to spot variations in crop health, chlorophyll absorption, and nitrogen requirements before they are visible to the naked human eye.

---

## 📐 Step 1: The Math Behind the Matrix

Satellite pixels do not represent simple colors; they represent raw physical reflection percentages. To turn these layers into actionable insights, we apply normalized difference equations.

1. **NDVI (Normalized Difference Vegetation Index):** Measures standard canopy greenness and leaf-area density.
$$NDVI = \frac{B08 - B04}{B08 + B04} \quad \left(\frac{\text{Near Infrared} - \text{Red}}{\text{Near Infrared} + \text{Red}}\right)$$

2. **NDRE (Normalized Difference Red Edge Index):** Uses the transitional "RedEdge" band. Because RedEdge light penetrates deeper into the crop canopy than standard Red light, NDRE is far superior for tracking dense, late-stage crops (like corn or wheat) where NDVI tends to oversaturate and lose sensitivity.
$$NDRE = \frac{B08 - B05}{B08 + B05} \quad \left(\frac{\text{Near Infrared} - \text{RedEdge}}{\text{Near Infrared} + \text{RedEdge}}\right)$$

* **Action Required:** Run the cell below to load your baseline packages and prepare the interactive analytics framework.

---

## 📦 Dependencies

In [1]:
# Uncomment the line below if you need to install the dependencies in your environment
# !pip install pystac-client pystac shapely planetary-computer requests matplotlib

import json
from pystac_client import Client
import planetary_computer as pc
from shapely.geometry import shape, mapping
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
import rioxarray

print("📦 Libraries imported successfully! Ready to connect to the datacube.")


Bad key keymap.all_axes in file matplotlibrc, line 398 ('keymap.all_axes : a                 # enable all axes')
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.10.9/lib/matplotlib/mpl-data/matplotlibrc
or from the matplotlib source distribution


📦 Libraries imported successfully! Ready to connect to the datacube.


---

## 🗺️ Area of Interest (AOI)

In [2]:
# Define a bounding box around a high-production agricultural region
# Format: [min_longitude, min_latitude, max_longitude, max_latitude]
aoi_bbox = [13.15, 52.35, 13.18, 52.38] 

# Convert the bounding box into a standard GeoJSON geometry dictionary
aoi_geometry = {
    "type": "Polygon",
    "coordinates": [[
        [aoi_bbox[0], aoi_bbox[1]],
        [aoi_bbox[0], aoi_bbox[3]],
        [aoi_bbox[2], aoi_bbox[3]],
        [aoi_bbox[2], aoi_bbox[1]],
        [aoi_bbox[0], aoi_bbox[1]]
    ]]
}

print("Farm boundary geometry locked in. Ready to query.")

Farm boundary geometry locked in. Ready to query.


---

## 🔍 Querying the STAC API

In [3]:
# Connect to the global cloud catalog endpoint
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

# Execute the programmatic search query
search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=aoi_geometry,
    datetime="2025-04-01/2025-04-10",
    query={"eo:cloud_cover": {"lt": 10}} # Filter: less than 10% cloud cover
)

# Fetch all matching items found in the cloud registry
items = list(search.get_items())
print(f"📡 API Query Complete! Found {len(items)} cloud-free satellite scenes matching your criteria.")

s2_items = [pc.sign(item) for item in search.get_items()]
cropping_poly = shape(aoi_geometry)

print(f"📈 Analytics engine armed. Ready to process {len(s2_items)} multi-spectral scenes.")

/usr/local/lib/python3.10/site-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


📡 API Query Complete! Found 4 cloud-free satellite scenes matching your criteria.
📈 Analytics engine armed. Ready to process 4 multi-spectral scenes.


---

## 🎛️ Step 2: Interactive Agronomic Index Generator

We will now build a dynamic processor. When you change the index option in the dropdown menu, the engine will query the precise raw bands needed from the cloud, crop them down to your exact coordinates, execute the array arithmetic, and plot an agronomic heatmap.

* **Action Required:** Execute this block to launch the interactive index workbench.

In [4]:
def process_agronomic_index(index_frame, index_type):
    if not s2_items:
        print("❌ No imagery layers found.")
        return
        
    current_item = s2_items[index_frame]
    print(f"⚡ Processing {index_type} array for target scene: {current_item.id}")
    
    try:
        # Determine which spectral assets are required for our selected calculation
        if index_type == "NDVI (Normalized Difference Vegetation Index)":
            needed_bands = ["B08", "B04"]  # Near-Infrared (NIR) & Red
        elif index_type == "NDRE (Nitrogen / Dense Canopy)":
            needed_bands = ["B08", "B05"]  # Near-Infrared (NIR) & RedEdge (B5)
            
        # Stream and clip the primary array (NIR - Band 8)
        url_nir = current_item.assets[needed_bands[0]].href
        raster_nir = rioxarray.open_rasterio(url_nir)
        clip_nir = raster_nir.rio.clip([cropping_poly], crs="EPSG:4326", from_disk=True)
        nir_array = clip_nir.data[0].astype(float)
        
        # Stream and clip the secondary array (Red or RedEdge)
        url_sec = current_item.assets[needed_bands[1]].href
        raster_sec = rioxarray.open_rasterio(url_sec)
        clip_sec = raster_sec.rio.clip([cropping_poly], crs="EPSG:4326", from_disk=True)
        sec_array = clip_sec.data[0].astype(float)
        
        # 🧮 Execute Matrix Algebra
        # We add 1e-5 to the denominator to prevent "Division by Zero" crashes over open water or shadows
        calculated_index = (nir_array - sec_array) / (nir_array + sec_array + 1e-5)
        
        # Clip theoretical output boundaries to valid index ranges [-1.0 to 1.0]
        calculated_index = np.clip(calculated_index, -1.0, 1.0)
        
        # 🎨 Render the Heatmap
        plt.figure(figsize=(11, 9))
        
        # Use 'RdYlGn' (Red-Yellow-Green) colormap. Low index = Red (soil/stress), High index = Deep Green (dense leaves)
        img_plot = plt.imshow(calculated_index, cmap="RdYlGn", vmin=0.0, vmax=1.0)
        
        # Attach a functional measurement colorbar to gauge actual field performance values
        cbar = plt.colorbar(img_plot, fraction=0.046, pad=0.04)
        cbar.set_label(f"Calculated {index_type} Absolute Value", rotation=270, labelpad=15, fontsize=10)
        
        plt.title(f"📊 Agronomic Assessment Layer: {index_type}\n📅 Date: {str(current_item.datetime)[:16]}", fontsize=12, fontweight='bold')
        plt.axis("off")
        plt.show()
        
    except Exception as e:
        print(f"💥 Matrix algebra execution failure: {e}")
        print("💡 Hint: Ensure you are choosing a Sentinel-2 scene with valid matching band footprints.")

# Build the interactive analytics interface cockpit
if s2_items:
    widgets.interact(
        process_agronomic_index,
        index_frame=widgets.IntSlider(min=0, max=len(s2_items)-1, step=1, value=0, description='Timeline:'),
        index_type=widgets.Dropdown(
            options=["NDVI", "NDRE (Nitrogen / Dense Canopy)"],
            value="NDVI",
            description="Index Model:"
        )
    )

interactive(children=(IntSlider(value=0, description='Timeline:', max=3), Dropdown(description='Index Model:',…

---

## ✅ Fixed pixel problem between differnet bands of Sentinel-2!

In [5]:
def process_agronomic_index(index_frame, index_type):
    if not s2_items:
        print("❌ No imagery layers found.")
        return
        
    current_item = s2_items[index_frame]
    print(f"⚡ Processing {index_type} array for target scene: {current_item.id}")
    
    try:
        # Determine which spectral assets are required for our selected calculation
        if index_type == "NDVI":
            needed_bands = ["B08", "B04"]  # NIR (10m) & Red (10m) -> Match perfectly natively
        elif index_type == "NDRE (Nitrogen / Dense Canopy)":
            needed_bands = ["B08", "B05"]  # NIR (10m) & RedEdge (20m) -> Requires resolution matching
            
        # 1. Stream and clip the baseline array (NIR - Band 8, 10m resolution)
        url_nir = current_item.assets[needed_bands[0]].href
        raster_nir = rioxarray.open_rasterio(url_nir)
        clip_nir = raster_nir.rio.clip([cropping_poly], crs="EPSG:4326", from_disk=True)
        
        # 2. Stream and clip the secondary array (Red or RedEdge)
        url_sec = current_item.assets[needed_bands[1]].href
        raster_sec = rioxarray.open_rasterio(url_sec)
        clip_sec = raster_sec.rio.clip([cropping_poly], crs="EPSG:4326", from_disk=True)
        
        # 📐 CRITICAL RESOLUTION ALIGNMENT STEP
        # If calculating NDRE, force the 20m RedEdge grid to match the 10m NIR grid matrix shape
        if index_type == "NDRE (Nitrogen / Dense Canopy)":
            clip_sec = clip_sec.rio.reproject_match(clip_nir)
            
        # Extract the raw aligned 2D arrays
        nir_array = clip_nir.data[0].astype(float)
        sec_array = clip_sec.data[0].astype(float)
        
        # 🧮 Execute Matrix Algebra safely
        calculated_index = (nir_array - sec_array) / (nir_array + sec_array + 1e-5)
        calculated_index = np.clip(calculated_index, -1.0, 1.0)
        
        # 🎨 Render the Heatmap
        plt.figure(figsize=(11, 9))
        img_plot = plt.imshow(calculated_index, cmap="RdYlGn", vmin=0.0, vmax=1.0)
        
        cbar = plt.colorbar(img_plot, fraction=0.046, pad=0.04)
        cbar.set_label(f"Calculated {index_type} Absolute Value", rotation=270, labelpad=15, fontsize=10)
        
        plt.title(f"📊 Aligned Agronomic Assessment: {index_type}\n📅 Date: {str(current_item.datetime)[:16]}", fontsize=12, fontweight='bold')
        plt.axis("off")
        plt.show()
        
    except Exception as e:
        print(f"💥 Matrix algebra execution failure: {e}")
        print("💡 Hint: Ensure you are choosing a Sentinel-2 scene with valid matching band footprints.")

# Build the interactive analytics interface cockpit
if s2_items:
    widgets.interact(
        process_agronomic_index,
        index_frame=widgets.IntSlider(min=0, max=len(s2_items)-1, step=1, value=0, description='Timeline:'),
        index_type=widgets.Dropdown(
            options=["NDVI", "NDRE (Nitrogen / Dense Canopy)"],
            value="NDVI",
            description="Index Model:"
        )
    )

interactive(children=(IntSlider(value=0, description='Timeline:', max=3), Dropdown(description='Index Model:',…

---

## 🛠️ Student Verification Checkpoint & Lab Challenge

Now that you can calculate real-world plant performance values across time, complete the following investigation:

1. **Spot the Saturation Difference:** Find a date late in the spring timeline where the crops have grown exceptionally thick. Toggle between **NDVI** and **NDRE**. Notice how the NDVI map turns into a flat, solid dark green block (saturates), while the **NDRE** map continues to display clear variations and structural patterns within the fields.
2. **Interpret the Value Scale:** Look closely at the side colorbar scales. What index values correspond to bare, plowed dirt fields versus active, dense crops?

---

### 🛠️ Next Up: **SF_Day01_part6.ipynb**